# Cleaning 5.1 - Clean tips (from calculator)

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import load_workbook
import xlwings as xw
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from create_variable_from_survey import create_var

In [2]:
# Load data
labs = pd.read_csv(config.EL_RAW_SAMPLE / "final_sample_with_EL_file_status.csv")

# Calculators folder
calculators_folder = config.CALCULATORS_WITH_TIPS

In [3]:
# Load survey dictionaries (helper for mapping variables to survey files, sheets and cells)
equip_mappings = pd.read_excel(config.SURVEY_DICTIONARIES / "helper_survey_dictionary.xlsx", sheet_name="Equipment")

# Calculator equipment types
calculator_equip = equip_mappings[equip_mappings["Equipment type"] != "fc"]

In [4]:
# Email confirmation only labs
email_confirmation = pd.read_excel(config.WAVE1_ENUMERATORS / "EL_visits_completed.xlsx", sheet_name = "No visit email conf")
labs["el_email_conf"] = labs["labgroupid"].isin(email_confirmation["labgroupid"]).astype(bool)

# Recovered data (technical issues etc.)
recovered_data = pd.read_excel(config.WAVE1_ENUMERATORS / "EL_visits_completed.xlsx", sheet_name = "Recovered")
labs["recovered_data"] = labs["labgroupid"].isin(recovered_data["labgroupid"]).astype(bool)
# Inidicator to replace EL data with missing for awareness, attitudes, checklist (if "To do" = "Replace EL qs with missing")
labs["replace_el_with_missing"] = labs["labgroupid"].isin(
    recovered_data[recovered_data["To do"] == "Replace EL qs with missing"]["labgroupid"]).astype(bool)

## (1) Combine excel files data into panel dataset (1 row per labgroupid x tip)

In [5]:
# List of labgroupids to process (all)
labgroupids = labs["labgroupid"].tolist()

labs = labs.copy()

# List of treated labgroupids
treated_labs = labs[labs["Treatment Status"] == "treatment"].copy()
labgroupids_t_only = treated_labs["labgroupid"].tolist()

# List of labs with EL data collected
el_done_labs = labs[labs["el_awareness_filled"] == True].copy()
labgroupids_el_done = el_done_labs["labgroupid"].tolist()

# List of labs to replace EL data with missing (from recovered data sheet)
replace_el_with_missing_labs = labs[labs["replace_el_with_missing"] == True].copy()
labgroupids_replace_el_missing = replace_el_with_missing_labs["labgroupid"].tolist()

# List of labs we are aware have no BL checklist
no_bl_checklists = pd.read_excel(config.WAVE1_LABS_LIST / "labs_no_BL_SPARK_checklist.xlsx")
labgroupids_no_bl_checklist = no_bl_checklists["labgroupid"].tolist()

In [6]:
# Extract tips from calculators (have to use xlwings as not loaded previously)

missing_calculator = []
tips_extracted = pd.DataFrame()

for labgroupid in labgroupids:

    mask = labs["labgroupid"] == labgroupid
    if not mask.any():
        continue

    calculator_path = calculators_folder / str(labgroupid) / "Energy_Use_Report.xlsx"
    if not calculator_path.exists():
        display(f"Missing calculator for labgroup {labgroupid}.")
        missing_calculator.append(labgroupid)
        continue

    wb = xw.Book(calculator_path)

    # Open "Tips" sheet
    ws = wb.sheets["Tips"]
    # Extract all non-missing cells
    raw_value = ws.range("A1").expand().value
    if not isinstance(raw_value, list):
        raw_value = [raw_value]
    non_missing_cells = [cell for cell in raw_value if cell is not None]

    # Create a DataFrame from the non-missing cells
    df = pd.DataFrame(non_missing_cells)
    # Rename the column to "tip"
    df.columns = ["tip"]
    # Add a column for the labgroupid
    df["labgroupid"] = labgroupid

    # Append the DataFrame to the main DataFrame
    tips_extracted = pd.concat([tips_extracted, df], ignore_index=True)

    # Close the workbook
    wb.close()

# Check that no groups missing calculator
assert not missing_calculator

In [7]:
# Print the first few rows of the tips DataFrame
print(tips_extracted["tip"].head(20))

0                                 Fridges (4 degrees C)
1     For type 1: You're opening the door a lot! Hav...
2     For type 2: You're opening the door a lot! Hav...
3     For type 3: You're opening the door a lot! Hav...
4                              Freezers (-20 degrees C)
5     For type 1: You're opening the door a lot! Hav...
6     For type 2: If you can warm up to -20C you can...
7     For type 2: You're opening the door a lot! Hav...
8     For type 3: You're opening the door a lot! Hav...
9                                        CO2 Incubators
10    For type 1: This unit is on a lot! If you only...
11    For type 2: This unit is on a lot! If you only...
12           There are no tips to present at this time.
13           There are no tips to present at this time.
14                                Fridges (4 degrees C)
15    For type 1: You're opening the door a lot! Hav...
16    For type 2: You're opening the door a lot! Hav...
17    For type 3: You're opening the door a lot!

## (2) Clean the tips dataset

In [8]:
tips = tips_extracted.copy()

# Create column "equipment"
# Header rows are the ones that are NOT a "For type X:" tip - these name the equipment for the tips below them
is_equipment_header = ~tips["tip"].str.match(r"^For type \d+", na=False)

tips["equipment"] = tips["tip"].where(is_equipment_header)
tips["equipment"] = tips.groupby("labgroupid")["equipment"].ffill()

# tips.head(20)

In [9]:
# Create column "type_no"
tips["type_no"] = tips["tip"].str.extract(r"For type (\d+)", expand=False)

In [10]:
# Drop all rows that have a missing "type_no" and are not "There are no tips to present at this time."
tips = tips[~(tips["type_no"].isna() & (tips["tip"] != "There are no tips to present at this time."))]

# Get rid of the "For type X: " part of the tip text for rows that have a type_no
tips["tip"] = tips.apply(lambda row: row["tip"].replace(f"For type {row['type_no']}: ", ""), axis=1)

# Get rid of the "For type X" part of the tip text for rows that have a type_no (colon missing)
tips["tip"] = tips.apply(lambda row: row["tip"].replace(f"For type {row['type_no']}", ""), axis=1)

# Replace equipment with "" for tips that are "There are no tips to present at this time."
tips.loc[tips["tip"] == "There are no tips to present at this time.", "equipment"] = ""

In [11]:
# Check the frequency of tips
print(tips["tip"].value_counts())

tip
There are no tips to present at this time.                                                                                                                                                70
Don't let that icing get out of hand! If you can de-ice your freezer (once every 6-12 months) you will keep it energy efficient.                                                          69
You're opening the door a lot! Have you used a fridge map or inventory to help you find your stuff? If you can reduce your door openings to 8 per day you can save 3.25 kWh per year       9
This unit is on a lot! If you only used this 255 days per year you would save 142.12 kWh per year.                                                                                         9
This unit is on a lot! If you only used this 255 days per year you would save 157.79 kWh per year.                                                                                         7
                                                   

In [12]:
# Check the frequency of tips and equipment combinations
print(tips.groupby(["tip", "equipment"]).size()
      .reset_index(name='count').sort_values(by='count', ascending=False))

                                                   tip  \
71          There are no tips to present at this time.   
0    Don't let that icing get out of hand! If you c...   
172  You're opening the door a lot! Have you used a...   
87   This unit is on a lot! If you only used this 2...   
88   This unit is on a lot! If you only used this 2...   
..                                                 ...   
95   This unit is on a lot! If you only used this 2...   
96   This unit is on a lot! If you only used this 2...   
21   If you can warm up to -20C you can save 216.81...   
98   This unit is on a lot! If you only used this 2...   
108  This unit is on a lot! Using for 6 hours a day...   

                    equipment  count  
71                                70  
0    Freezers (-20 degrees C)     69  
172     Fridges (4 degrees C)      9  
87             CO2 Incubators      9  
88             CO2 Incubators      7  
..                        ...    ...  
95             CO2 Incubators 

In [13]:
print(tips["equipment"].value_counts())

equipment
Freezers (-20 degrees C)     178
                              70
Fridges (4 degrees C)         64
ULT Freezers                  40
CO2 Incubators                39
Water Baths                   17
Glassware Drying Cabinets     10
Microbio Safety Cabinets       6
Name: count, dtype: int64


In [14]:
# Clean the equipment names to match the other data
equipment_names_mapping = {
    "Freezers (-20 degrees C)": "Freezer",
    "Fridges (4 degrees C)": "Fridge",
    "ULT Freezers": "ULT freezer",
    "CO2 Incubators": "CO2 incubator",
    "Water Baths": "Water bath",
    "Glassware Drying Cabinets": "Glassware drying cabinet",
    "Microbio Safety Cabinets": "Microbiological safety cabinet"
}

for old_name, new_name in equipment_names_mapping.items():
    tips["equipment"] = tips["equipment"].str.replace(old_name, new_name, regex=False)

In [15]:
# Check that for labgroupids that have the tip "There are no tips to present at this time." there is only one observation per labgroupid
labgroupids_with_no_tips = tips[tips["tip"] == "There are no tips to present at this time."]["labgroupid"].unique()
labgroupids_with_tips = tips[tips["tip"] != "There are no tips to present at this time."]["labgroupid"].unique()

# check that labgroupids_with_no_tips and labgroupids_with_tips are disjoint
assert set(labgroupids_with_no_tips).isdisjoint(set(labgroupids_with_tips))

# Check number of unique labgroupids with no tips and with tips
print(f"Number of unique labgroupids with no tips: {len(labgroupids_with_no_tips)}")
print(f"Number of unique labgroupids with tips: {len(labgroupids_with_tips)}")

# Check number of labgroupids in the tips dataframe
print(f"Number of unique labgroupids in the tips dataframe: {tips['labgroupid'].nunique()}")

# Check which labgroupid is in labgroupids but not in tips dataframe
labgroupids_not_in_tips = set(labgroupids) - set(tips["labgroupid"].unique())
# print(f"Labgroupids not in tips dataframe: {labgroupids_not_in_tips}")

# For this lab create a row with the tip "There are no tips to present at this time." and an empty equipment/type_no column
for labgroupid in labgroupids_not_in_tips:
    new_row = pd.DataFrame({
        "labgroupid": [labgroupid],
        "tip": ["There are no tips to present at this time."],
        "equipment": [""],
        "type_no": [""],
        "tips_error": [True]
    })

    tips = pd.concat([tips, new_row], ignore_index=True)

# Check that no labgroupids are missing from the tips dataframe
assert set(labgroupids) == set(tips["labgroupid"].unique())

Number of unique labgroupids with no tips: 70
Number of unique labgroupids with tips: 67
Number of unique labgroupids in the tips dataframe: 137


In [16]:
# Save this labgroupid that is not in the tips dataframe to a csv file
missing_tips = pd.DataFrame({"labgroupid": list(labgroupids_not_in_tips)})
missing_tips.to_csv(config.PROCESSED_DATA / "labgroupids_tips_error.csv", index=False)

In [17]:
# Save cleaned dataset
tips.to_csv(config.CLEAN_DATA / "tips_cleaned.csv", index = False)